# S&P 500 Stock Price Movement Predictor: Exploratory Data Analysis (EDA)

This notebook explores the cleaned historical dataset (`data/processed/SP500_Cleaned_Data.parquet`).
It examines dataset distributions, returns, volume dynamics, and outliers to inform our feature engineering and LSTM architecture.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_PATH = Path("../data/processed/SP500_Cleaned_Data.parquet")
df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} rows across {len(df.columns)} columns.")
df.head()

## 1. Dataset Dimensions & Coverage

In [ ]:
print(f"Unique Tickers:  {df['Ticker'].nunique()}")
print(f"Date Range:      {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Trading Days:    {df['Date'].nunique():,} unique sessions")

ticker_counts = df['Ticker'].value_counts()
print(f"Median Days/Stock: {ticker_counts.median():.0f}")
print(f"Full-history Tickers (6,573 days): {(ticker_counts == 6573).sum()} ({(ticker_counts == 6573).mean()*100:.1f}%)")

## 2. Daily Price Returns Distribution & Fat Tails

In [ ]:
df['Return'] = df.groupby('Ticker')['Close'].pct_change()
returns = df['Return'].dropna()

print(f"Mean Daily Return:   {returns.mean() * 100:.4f}%")
print(f"Std Dev Return:      {returns.std() * 100:.4f}%")
print(f"Skewness:            {stats.skew(returns):.4f}")
print(f"Excess Kurtosis:     {stats.kurtosis(returns):.2f} (Extreme fat tails)")

## 3. Visualizing Representative Stocks (AAPL, MSFT, JPM, JNJ, CAT)

In [ ]:
rep_tickers = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'CAT']
fig, ax = plt.subplots(figsize=(12, 5))
for ticker in rep_tickers:
    sub = df[df['Ticker'] == ticker]
    ax.plot(sub['Date'], sub['Close'] / sub['Close'].iloc[0] * 100, label=ticker)
ax.set_yscale('log')
ax.set_title('Normalized Historical Growth (Base 100, Log Scale)')
ax.set_ylabel('Normalized Price')
ax.legend()
plt.show()

## 4. Volume vs Price Movement (|Return|)

In [ ]:
valid_sub = df.dropna(subset=['Return']).copy()
valid_sub = valid_sub[valid_sub['Volume'] > 0]
valid_sub['Abs_Return'] = valid_sub['Return'].abs()
valid_sub['Log_Volume'] = np.log10(valid_sub['Volume'])

print(f"Pearson Correlation (|Return| vs Log Volume): {valid_sub['Abs_Return'].corr(valid_sub['Log_Volume']):.4f}")
print(f"Spearman Correlation (|Return| vs Volume):     {valid_sub['Abs_Return'].corr(valid_sub['Log_Volume'], method='spearman'):.4f}")